# 05. 관제용 모델 저장 (구역 하나씩)

| 항목 | 내용 |
| --- | --- |
| 바꿀 곳 | 1번 셀의 `DATA_FILE`, `ZONE` 두 줄. 입력 컬럼 묶음(`USE_`)은 03과 같은 값으로 둠 |
| 입력 | `data/processed/cal01_base_v6.csv`, 03이 저장한 `model_results/*_summary.csv`, `*_predictions.csv` |
| 출력 | `models/<라벨 버전>/<입력 태그>/<구역>/` 안의 모델 파일과 설명 파일, `models/registry.json` |
| 이 노트북이 하는 일 | 03과 같은 입력 계산으로 배포용 모델을 학습해 파일로 남김. 03은 시험할 때마다 모델을 버리므로 따로 만듦 |

**만드는 것** (구역 하나당)

| 파일 | 내용 |
| --- | --- |
| `rf-all14-<날짜>.joblib` | RandomForest, 14 LOT 전체로 학습. 관제에 올리는 것 |
| `rf-fold13-<날짜>.joblib` | RandomForest, LOT 1 ~ 13으로 학습. 03의 마지막 시험과 같은 모델이라 재현 확인에 씀 |
| `iso-all14-<날짜>.joblib` | IsolationForest, 14 LOT 전체 |
| `features.json` | 입력 컬럼 순서와 각 컬럼을 만드는 규칙 (관제가 같은 계산을 하기 위함) |
| `config.json` | 스위치, 파라미터, 학습 LOT, 라벨 규칙, 데이터 파일 해시, 재현 확인 결과 |
| `metrics.json` | 03 시간순 걷기 결과 (F1, 정밀도, 재현율, 헛경보) |
| `fold13_expected.json` | fold13 모델이 LOT 133을 예측한 값. 관제 쪽 입력 계산이 맞는지 확인할 때 씀 |

- 모델 이름: `cal01-<라벨버전>-<입력태그>-<구역>-<모델>-<학습범위>-<날짜>`. 입력 태그 `final`은 `common_roll10_zpast_opcommon_furnace_control`
- `models/`는 git에 올리지 않음 (README만). 관제 저장소의 `scripts/import_models.py`가 복사해 감
- Z-score는 파일이 없는 규칙이라 `models/<라벨 버전>/zscore.json`에 설정만 적어 함께 등록함


## 1. 공통 경로와 설정 (`DATA_FILE`, `ZONE` 두 줄만 바꿈)

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v6)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v6.csv"  # cal01_base_v1.csv ~ cal01_base_v6.csv
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 비교할 구역
ZONE = "Z4"  # CLN, Z1 ~ Z9, COOLING

# 입력 컬럼 묶음 (v6 입력은 포화비율, 경과시간, 변화율만 꺼짐)
# 공통 움직임 컬럼 (Z3 ~ Z6 온도가 최근 5분 사이에 같이 움직였는지)을 넣을지
USE_COMMON = True

# 수치 컬럼 묶음 (모두 지난 값만 사용. 포화비율, 경과시간, 변화율은 v6 실험에서 효과가 없어 꺼 둠)
USE_SAT = False      # OP가 끝(99% 이상, 5% 이하)에 닿은 비율, 최근 10분
USE_ELAPSED = False  # LOT 첫 기록부터 지난 분
USE_ROLL10 = True   # 직전 10분 평균과의 차이, 직전 10분 표준편차
USE_RATE = False     # 최근 5분 동안 1분에 평균 얼마나 변했는지
USE_ZPAST = True    # 지난 값만으로 계산한 |Z| (Z-score 기준선과 같은 계산)
USE_OPCOMMON = True # Z3 ~ Z6 출력(OP)이 최근 5분 사이에 같이 움직였는지
USE_FURNACE = True  # Z1 ~ Z8 중 최근 5분 안에 같이 튄 구역 수
USE_CONTROL = True  # 최근 5분 동안 내 구역 TC와 OP가 같은 방향으로 움직인 비율

## 2. 라이브러리와 저장 폴더

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RESULT_DIR = os.path.join(PROCESSED_DIR, "model_results")
os.makedirs(RESULT_DIR, exist_ok=True)

# v4 ~ v6은 1분 묶음 데이터
MINUTE_DATA = DATA_FILE in ["cal01_base_v4.csv", "cal01_base_v5.csv", "cal01_base_v6.csv"]
print(DATA_FILE, "/ 1분 데이터:", MINUTE_DATA)

## 3. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

sensors = GROUPS[ZONE]
print(ZONE, "센서:", sensors)

# 공통 움직임 그룹: 같이 움직이는 구역끼리 묶음 (5분 변화 상관: Z1 → Z2 0.88 (2분 뒤), Z3 → Z4 0.51 (4분 뒤), Z4 ↔ Z5 0.62, Z7 ↔ Z8 0.19)
# CLN, Z9, COOLING은 다른 구역과 상관이 0.3 미만이라 그룹이 없음
COMMON_GROUPS = {
    "Z1": ["Z1", "Z2"], "Z2": ["Z1", "Z2"],                                   # 건조, 예열
    "Z3": ["Z3", "Z4", "Z5", "Z6"], "Z4": ["Z3", "Z4", "Z5", "Z6"],           # 가열, 균열
    "Z5": ["Z3", "Z4", "Z5", "Z6"], "Z6": ["Z3", "Z4", "Z5", "Z6"],
    "Z7": ["Z7", "Z8"], "Z8": ["Z7", "Z8"],                                   # 과시효
}

# 내 구역이 속한 그룹의 온도와 출력 (그룹이 없으면 빈 목록이라 공통 움직임 컬럼이 안 생김)
COMMON_SENSORS = []
COMMON_OPS = []
for zone in COMMON_GROUPS.get(ZONE, []):
    COMMON_SENSORS.append("TC-" + zone)
    COMMON_OPS.append("OP-" + zone)
print("공통 움직임 그룹:", COMMON_SENSORS)

# 로 전체 동시 이상 수에 쓰는 구역 (건조대 Z1, Z2부터 과시효대 Z7, Z8까지)
FURNACE_ZONES = ["Z1", "Z2", "Z3", "Z4", "Z5", "Z6", "Z7", "Z8"]

## 4. 데이터 읽기 (필요한 컬럼만)

In [ ]:
columns = ["LOT_NO", "MEAS_DT", "EXCLUDED", ZONE + "__STAT_LABEL", ZONE + "__DOMAIN_LABEL"]

for sensor in sensors:
    columns.append(sensor)

    if MINUTE_DATA:
        columns = columns + [sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]

# 공통 움직임 컬럼을 쓰면 Z3 ~ Z6 온도도 읽음
if USE_COMMON:
    for sensor in COMMON_SENSORS:
        if sensor not in columns:
            columns.append(sensor)

if USE_OPCOMMON:
    for sensor in COMMON_OPS:
        if sensor not in columns:
            columns.append(sensor)

if USE_FURNACE:
    for zone in FURNACE_ZONES:
        for sensor in GROUPS[zone]:
            if sensor not in columns:
                columns.append(sensor)

df = pd.read_csv(BASE_CSV, encoding=ENC, usecols=columns)
df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"], errors="raise")
df = df.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)

# LOT 첫 기록부터 지난 분 (USE_ELAPSED일 때 입력으로 씀)
lot_start = df.groupby("LOT_NO")["MEAS_DT"].transform("min")
df["LOT__ELAPSED"] = (df["MEAS_DT"] - lot_start).dt.total_seconds() / 60

print(df.shape)

## 5. LABEL 만들기

도메인 판단이 있으면 도메인 값, 없으면 통계 후보 값을 씀. 제외 행과 라벨이 빈 행은 뺌.

In [ ]:
df["LABEL"] = df[ZONE + "__DOMAIN_LABEL"].fillna(df[ZONE + "__STAT_LABEL"])

# 다른 규칙을 쓰려면 위 한 줄만 바꿈
# 예) 팀이 1로 확정한 것만 1로 볼 때
# df["LABEL"] = (df[ZONE + "__DOMAIN_LABEL"] == 1).astype(int).where(df[ZONE + "__STAT_LABEL"].notna())

df = df[(df["EXCLUDED"] == False) & df["LABEL"].notna()].reset_index(drop=True)
df["LABEL"] = df["LABEL"].astype(int)

print("평가할 행:", len(df))
print("LABEL 1 비율:", round(df["LABEL"].mean() * 100, 2), "%")

df.groupby("LOT_NO")["LABEL"].agg(["count", "sum"])

## 6. 입력 변수 만들기

| 변수 | 뜻 | 원기록 (v1 ~ v3) | 1분 (v4 ~ v6) |
| --- | --- | --- | --- |
| 센서 | 현재 값 | O | O (1분 평균) |
| `__DIFF` | 바로 앞 행과의 차이 | O | O |
| `__DEV60` | 직전 60행 평균과의 차이 | O | |
| `__STD60` | 직전 60행 표준편차 | O | |
| `__MIN`, `__MAX`, `__STD` | 1분 안의 최소, 최대, 표준편차 | | O |
| `온도__CHG5` | 내 구역 그룹(Z1·Z2, Z3~Z6, Z7·Z8) 온도의 5분 전 대비 변화 (`USE_COMMON`일 때) | O | O |
| `COMMON__OTHERS` | 내 구역 말고 같은 그룹 온도 중 최근 5분 안에 \|Z\| > 3이었던 수 (`USE_COMMON`일 때) | O | O |
| `OP__SAT10` | 최근 10분 중 OP가 99% 이상이나 5% 이하에 닿은 비율 (`USE_SAT`일 때) | O | O (1분 최대, 최소로 판단) |
| `LOT__ELAPSED` | LOT 첫 기록부터 지난 분 (`USE_ELAPSED`일 때) | O | O |
| `__DEV10`, `__STD10` | 직전 10분 평균과의 차이, 직전 10분 표준편차 (`USE_ROLL10`일 때) | O | O |
| `__RATE5` | 최근 5분 동안 1분에 평균 얼마나 변했는지 (`USE_RATE`일 때) | O | O |
| `__ZPAST` | 지난 값만으로 계산한 \|Z\| (`USE_ZPAST`일 때). 7번 Z-score 기준선과 같은 계산 | O | O |
| `__ZPAST_EDGE` | 1분 안의 최대, 최소 중 지난 평균에서 더 먼 쪽의 \|Z\| (`USE_ZPAST`일 때) | | O |
| `OP__CHG5`, `COMMON__OP_OTHERS` | 같은 그룹 출력의 5분 전 대비 변화, 내 구역 말고 최근 5분 안에 \|Z\| > 3이었던 출력 수 (`USE_OPCOMMON`일 때) | O | O |
| `FURNACE__OTHERS` | Z1 ~ Z8 중 내 구역 말고 최근 5분 안에 센서 하나라도 \|Z\| > 3이었던 구역 수 (`USE_FURNACE`일 때) | O | O |
| `CONTROL__SAME5` | 최근 5분 동안 내 구역 TC와 OP가 같은 방향으로 움직인 비율. 정상 제어면 반대 방향 (`USE_CONTROL`일 때) | O | O |

- 모든 계산은 같은 LOT 안에서, 현재보다 앞선 행만 사용함
- 공통 움직임의 \|Z\|도 7번 Z-score 기준과 같이 지난 값만으로 계산함
- 5분, 10분 전 값이 없는 LOT 앞부분은 0으로 둠 (컬럼을 켜고 꺼도 평가하는 행이 같도록)
- OP가 끝에 닿는 구역은 사실상 Z3뿐임 (OP-Z4 ~ Z6은 30 ~ 86% 안에서 움직임)

In [ ]:
features = []

for sensor in sensors:
    features.append(sensor)
    features.append(sensor + "__DIFF")

    if MINUTE_DATA:
        features = features + [sensor + "__MIN", sensor + "__MAX", sensor + "__STD"]
    else:
        features = features + [sensor + "__DEV60", sensor + "__STD60"]

lots = sorted(df["LOT_NO"].unique())

for lot in lots:
    rows = df["LOT_NO"] == lot

    for sensor in sensors:
        values = df.loc[rows, sensor]
        previous = values.shift(1)  # 바로 앞 행 값

        df.loc[rows, sensor + "__DIFF"] = values - previous

        if not MINUTE_DATA:
            df.loc[rows, sensor + "__DEV60"] = values - previous.rolling(60, min_periods=10).mean()
            df.loc[rows, sensor + "__STD60"] = previous.rolling(60, min_periods=10).std()

# 1분 안에 행이 하나뿐이면 표준편차가 비므로 0으로 채움
if MINUTE_DATA:
    for sensor in sensors:
        df[sensor + "__STD"] = df[sensor + "__STD"].fillna(0)

# 공통 움직임 컬럼: Z3 ~ Z6 온도가 최근 5분 사이에 같이 움직였는지
if USE_COMMON:
    WINDOW = 5 if MINUTE_DATA else 300  # 1분 데이터는 5행, 원기록은 300행 (둘 다 5분)

    for sensor in COMMON_SENSORS:
        features.append(sensor + "__CHG5")
    features.append("COMMON__OTHERS")

    df["COMMON__OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in COMMON_SENSORS:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            # 5분 전 대비 변화
            df.loc[rows, sensor + "__CHG5"] = (values - values.shift(WINDOW)).fillna(0)

            # 내 구역 온도는 세지 않음
            if sensor in sensors:
                continue

            # 지난 값만으로 |Z| > 3이었는지, 최근 5분 안에 한 번이라도
            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            over = ((values - past_mean).abs() / past_std > 3).astype(int)
            recent = over.rolling(WINDOW, min_periods=1).max()

            df.loc[rows, "COMMON__OTHERS"] = df.loc[rows, "COMMON__OTHERS"] + recent

# 1분 데이터는 10행, 원기록은 600행 (둘 다 10분)
WINDOW10 = 10 if MINUTE_DATA else 600

# OP 포화비율: 최근 10분 중 OP가 끝(99% 이상 또는 5% 이하)에 닿은 비율
if USE_SAT:
    op_sensors = []
    for sensor in sensors:
        if sensor.startswith("OP-"):
            op_sensors.append(sensor)
            features.append(sensor + "__SAT10")

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in op_sensors:
            if MINUTE_DATA:
                touched = (df.loc[rows, sensor + "__MAX"] >= 99) | (df.loc[rows, sensor + "__MIN"] <= 5)
            else:
                touched = (df.loc[rows, sensor] >= 99) | (df.loc[rows, sensor] <= 5)

            df.loc[rows, sensor + "__SAT10"] = touched.astype(int).rolling(WINDOW10, min_periods=1).mean()

# LOT 경과시간: 4번 셀에서 계산한 값을 입력에 넣음
if USE_ELAPSED:
    features.append("LOT__ELAPSED")

# 10분 이동통계: 직전 10분 평균과의 차이, 직전 10분 표준편차
if USE_ROLL10:
    for sensor in sensors:
        features = features + [sensor + "__DEV10", sensor + "__STD10"]

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in sensors:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            df.loc[rows, sensor + "__DEV10"] = (values - previous.rolling(WINDOW10, min_periods=2).mean()).fillna(0)
            df.loc[rows, sensor + "__STD10"] = previous.rolling(WINDOW10, min_periods=2).std().fillna(0)

# 변화율: 최근 5분 동안 1분에 평균 얼마나 변했는지
if USE_RATE:
    STEP = 5 if MINUTE_DATA else 300  # 5분 전 행

    for sensor in sensors:
        features.append(sensor + "__RATE5")

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in sensors:
            values = df.loc[rows, sensor]
            df.loc[rows, sensor + "__RATE5"] = ((values - values.shift(STEP)) / 5).fillna(0)

# 지난 값 |Z|: Z-score 기준선과 같은 정보를 RandomForest에도 줌
if USE_ZPAST:
    for sensor in sensors:
        features.append(sensor + "__ZPAST")
        if MINUTE_DATA:
            features.append(sensor + "__ZPAST_EDGE")

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in sensors:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            df.loc[rows, sensor + "__ZPAST"] = ((values - past_mean).abs() / past_std).fillna(0)

            # 1분 안의 최대, 최소 중 지난 평균에서 더 먼 쪽
            if MINUTE_DATA:
                high = (df.loc[rows, sensor + "__MAX"] - past_mean).abs()
                low = (df.loc[rows, sensor + "__MIN"] - past_mean).abs()
                edge = high.where(high > low, low)
                df.loc[rows, sensor + "__ZPAST_EDGE"] = (edge / past_std).fillna(0)

# OP 공통 움직임: Z3 ~ Z6 출력이 최근 5분 사이에 같이 움직였는지
if USE_OPCOMMON:
    WINDOW = 5 if MINUTE_DATA else 300

    for sensor in COMMON_OPS:
        features.append(sensor + "__CHG5")
    features.append("COMMON__OP_OTHERS")

    df["COMMON__OP_OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for sensor in COMMON_OPS:
            values = df.loc[rows, sensor]
            previous = values.shift(1)

            df.loc[rows, sensor + "__CHG5"] = (values - values.shift(WINDOW)).fillna(0)

            # 내 구역 출력은 세지 않음
            if sensor in sensors:
                continue

            past_mean = previous.expanding(min_periods=30).mean()
            past_std = previous.expanding(min_periods=30).std(ddof=0)
            past_std = past_std.replace(0, np.nan)

            over = ((values - past_mean).abs() / past_std > 3).astype(int)
            recent = over.rolling(WINDOW, min_periods=1).max()

            df.loc[rows, "COMMON__OP_OTHERS"] = df.loc[rows, "COMMON__OP_OTHERS"] + recent

# 로 전체 동시 이상 수: Z1 ~ Z8 중 내 구역 말고 최근 5분 안에 센서 하나라도 |Z| > 3이었던 구역 수
if USE_FURNACE:
    WINDOW = 5 if MINUTE_DATA else 300

    features.append("FURNACE__OTHERS")
    df["FURNACE__OTHERS"] = 0

    for lot in lots:
        rows = df["LOT_NO"] == lot

        for zone in FURNACE_ZONES:
            if zone == ZONE:
                continue

            # 이 구역 센서 중 하나라도 넘었으면 1
            zone_over = pd.Series(0, index=df.index[rows])

            for sensor in GROUPS[zone]:
                values = df.loc[rows, sensor]
                previous = values.shift(1)

                past_mean = previous.expanding(min_periods=30).mean()
                past_std = previous.expanding(min_periods=30).std(ddof=0)
                past_std = past_std.replace(0, np.nan)

                over = ((values - past_mean).abs() / past_std > 3).astype(int)
                zone_over = (zone_over + over).clip(upper=1)

            recent = zone_over.rolling(WINDOW, min_periods=1).max()
            df.loc[rows, "FURNACE__OTHERS"] = df.loc[rows, "FURNACE__OTHERS"] + recent

# TC-OP 제어 어긋남: 최근 5분 동안 내 구역 TC와 OP가 같은 방향으로 움직인 비율
if USE_CONTROL:
    WINDOW = 5 if MINUTE_DATA else 300

    tc_sensor = None
    op_sensor = None
    for sensor in sensors:
        if sensor.startswith("TC-"):
            tc_sensor = sensor
        if sensor.startswith("OP-"):
            op_sensor = sensor

    # TC와 OP가 함께 있는 구역만 (CLN, Z9, COOLING은 OP가 없음)
    if tc_sensor is not None and op_sensor is not None:
        features.append("CONTROL__SAME5")

        for lot in lots:
            rows = df["LOT_NO"] == lot

            tc_change = df.loc[rows, tc_sensor].diff()
            op_change = df.loc[rows, op_sensor].diff()

            same = ((tc_change * op_change) > 0).astype(int)
            df.loc[rows, "CONTROL__SAME5"] = same.rolling(WINDOW, min_periods=1).mean()

# LOT 앞부분처럼 값이 빈 행은 뺌
before = len(df)
df = df.dropna(subset=features).reset_index(drop=True)

print("입력 변수:", features)
print("값이 비어 뺀 행:", before - len(df))

## 7. 입력 묶음 이름과 저장 폴더

03과 같은 규칙으로 켠 묶음을 이름에 붙임. 모두 켠 v6 입력은 `final`로 줄여 씀

In [ ]:
import datetime
import hashlib
import json
import platform

import joblib
import sklearn

SWITCHES = [
    [USE_COMMON, "공통 움직임", "_common"],
    [USE_SAT, "포화비율", "_sat"],
    [USE_ELAPSED, "경과시간", "_elapsed"],
    [USE_ROLL10, "10분 통계", "_roll10"],
    [USE_RATE, "변화율", "_rate"],
    [USE_ZPAST, "지난 값 Z", "_zpast"],
    [USE_OPCOMMON, "OP 공통 움직임", "_opcommon"],
    [USE_FURNACE, "로 전체 동시 이상", "_furnace"],
    [USE_CONTROL, "제어 어긋남", "_control"],
]

name_tail = ""
switch_values = {}
for on, label, tail in SWITCHES:
    switch_values[tail.strip("_")] = bool(on)
    if on:
        name_tail = name_tail + tail

if name_tail == "_common_roll10_zpast_opcommon_furnace_control":
    input_tag = "final"
elif name_tail == "":
    input_tag = "base"
else:
    input_tag = name_tail.strip("_")

LABEL_VERSION = DATA_FILE.replace("cal01_base_", "").replace(".csv", "")   # v6
TODAY = datetime.date.today().strftime("%Y%m%d")

MODEL_DIR = os.path.join(ROOT, "models")
VERSION_DIR = os.path.join(MODEL_DIR, LABEL_VERSION)
ZONE_DIR = os.path.join(VERSION_DIR, input_tag, ZONE)
os.makedirs(ZONE_DIR, exist_ok=True)

# 03이 저장한 결과 이름 (같은 구역, 같은 묶음)
result_name = DATA_FILE.replace(".csv", "") + "_" + ZONE + name_tail

# 구역마다 정답에 어떤 팀 판단이 들어갔는지 (registry의 근거 단계)
EVIDENCE = {
    "Z3": "팀 판단 1과 0 있음: 공통 움직임 해제(v2, v3)와 팀 판단(v6 T, S)",
    "Z4": "팀 판단 1과 0 있음: 공통 움직임 해제(v2, v3)와 팀 판단(v6 T, S)",
    "Z5": "팀 판단 0만 있음: 공통 움직임 해제(v2, v3)와 전 구역 검토(v6 S). 1은 통계 후보",
    "Z6": "팀 판단 0만 있음: 공통 움직임 해제(v2, v3)와 전 구역 검토(v6 S). 1은 통계 후보",
    "Z1": "팀 판단 0만 있음: 공통 움직임 해제(v3 E)와 전 구역 검토(v6 S). 1은 통계 후보",
    "Z2": "팀 판단 0만 있음: 공통 움직임 해제(v3 E)와 전 구역 검토(v6 S). 1은 통계 후보",
    "Z7": "팀 판단 0만 있음: 전 구역 검토(v6 S). 1은 통계 후보",
    "Z8": "팀 판단 0만 있음: 전 구역 검토(v6 S). 1은 통계 후보",
}

# 모델 이름
MODEL_IDS = {
    "rf_all14": "cal01-" + LABEL_VERSION + "-" + input_tag + "-" + ZONE + "-rf-all14-" + TODAY,
    "rf_fold13": "cal01-" + LABEL_VERSION + "-" + input_tag + "-" + ZONE + "-rf-fold13-" + TODAY,
    "iso_all14": "cal01-" + LABEL_VERSION + "-" + input_tag + "-" + ZONE + "-iso-all14-" + TODAY,
}

print("입력 태그:", input_tag, "/ 저장 폴더:", ZONE_DIR)
print("03 결과 이름:", result_name)

## 8. 학습

| 모델 | 학습 LOT | 용도 |
| --- | --- | --- |
| RandomForest all14 | 14개 전부 | 관제 배포 |
| RandomForest fold13 | 앞 13개 | 03의 마지막 시험(LOT 133)과 같은 조건. 재현 확인 |
| IsolationForest all14 | 14개 전부 | 관제의 검토 후보 표시 |

- 파라미터와 `random_state=42`는 03과 같음
- IsolationForest의 `contamination`은 03처럼 학습 자료의 LABEL 1 비율

In [ ]:
lots = sorted(df["LOT_NO"].unique())

X_all = df[features]
y_all = df["LABEL"]
rate_all = y_all.mean()

rf_all14 = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
rf_all14.fit(X_all, y_all)

iso_all14 = IsolationForest(n_estimators=100, contamination=min(rate_all, 0.5), random_state=42, n_jobs=-1)
iso_all14.fit(X_all)

train13 = df["LOT_NO"].isin(lots[:13])
rf_fold13 = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
rf_fold13.fit(df.loc[train13, features], df.loc[train13, "LABEL"])

print("학습 행:", len(df), "/ LABEL 1 비율:", round(rate_all * 100, 2), "% / 입력 컬럼:", len(features))
print("fold13 학습 LOT:", [int(x) for x in lots[:13]])

## 9. 재현 확인: fold13 모델이 LOT 133을 03과 똑같이 예측하는가

03에서 LOT 133을 시험할 때 쓴 모델과 같은 조건이므로 1분 단위 예측이 모두 같아야 함. 다르면 이 노트북의 입력 계산이 03과 어긋난 것

In [ ]:
RESULT_DIR = os.path.join(PROCESSED_DIR, "model_results")
saved = pd.read_csv(os.path.join(RESULT_DIR, result_name + "_predictions.csv"), encoding=ENC)
saved["MEAS_DT"] = pd.to_datetime(saved["MEAS_DT"], errors="raise")

last_lot = lots[13]
last = df[df["LOT_NO"] == last_lot].copy()
last["PRED_05"] = rf_fold13.predict(last[features])

compare = last[["MEAS_DT", "PRED_05"]].merge(saved[saved["LOT_NO"] == last_lot][["MEAS_DT", "PRED_RF"]], on="MEAS_DT", how="left")
same_minutes = int((compare["PRED_05"] == compare["PRED_RF"]).sum())
total_minutes = len(compare)
reproduced = same_minutes == total_minutes

print("LOT", last_lot, "/ 같은 분:", same_minutes, "/", total_minutes, "/ 재현:", reproduced)

## 10. 저장: 모델 파일, features.json, config.json, metrics.json, fold13_expected.json

In [ ]:
# 입력 컬럼마다 만드는 규칙 (관제가 같은 계산을 하도록). 모든 계산은 같은 LOT 안의 지난 값만 씀
feature_rules = []
for name in features:
    if name in sensors:
        rule = "1분 평균 현재값 (cal01_base의 센서 컬럼)"
    elif name.endswith("__DIFF"):
        rule = "바로 앞 1분 값과의 차이"
    elif name.endswith("__MIN") or name.endswith("__MAX") or name.endswith("__STD"):
        rule = "1분 안 원기록의 최소, 최대, 표준편차 (cal01_base 집계 컬럼. 표준편차 빈칸은 0)"
    elif name.startswith("OP-") and name.endswith("__CHG5"):
        rule = "Z3 ~ Z6 출력의 5분 전 값 대비 변화 (앞 5분은 0)"
    elif name.endswith("__CHG5"):
        rule = "Z3 ~ Z6 온도의 5분 전 값 대비 변화 (앞 5분은 0)"
    elif name == "COMMON__OTHERS":
        rule = "내 구역 말고 Z3 ~ Z6 온도 중 최근 5분 안에 지난 값 |Z| > 3이었던 수 (0 ~ 3)"
    elif name == "COMMON__OP_OTHERS":
        rule = "내 구역 말고 Z3 ~ Z6 출력 중 최근 5분 안에 지난 값 |Z| > 3이었던 수"
    elif name == "FURNACE__OTHERS":
        rule = "Z1 ~ Z8 중 내 구역 말고 최근 5분 안에 센서 하나라도 지난 값 |Z| > 3이었던 구역 수"
    elif name == "CONTROL__SAME5":
        rule = "최근 5분 동안 내 구역 TC와 OP의 1분 변화가 같은 부호인 비율"
    elif name.endswith("__DEV10"):
        rule = "직전 10분 평균과의 차이"
    elif name.endswith("__STD10"):
        rule = "직전 10분 표준편차"
    elif name.endswith("__ZPAST_EDGE"):
        rule = "1분 최대, 최소 중 지난 평균에서 더 먼 쪽과 지난 평균의 차이를 지난 표준편차로 나눈 값"
    elif name.endswith("__ZPAST"):
        rule = "|현재값 - 지난 평균| / 지난 표준편차. 지난 평균과 표준편차는 같은 LOT에서 현재 분 이전까지의 expanding(min_periods=30), ddof=0. 앞 30분은 0"
    elif name.endswith("__SAT10"):
        rule = "최근 10분 중 OP가 99% 이상이나 5% 이하에 닿은 비율"
    elif name == "LOT__ELAPSED":
        rule = "LOT 첫 기록부터 지난 분"
    elif name.endswith("__RATE5"):
        rule = "최근 5분 동안 1분에 평균 얼마나 변했는지"
    else:
        rule = "03 노트북 6번 셀 참고"
    feature_rules.append({"name": name, "rule": rule})

features_doc = {
    "zone": ZONE,
    "zone_sensors": sensors,
    "feature_order": features,
    "features": feature_rules,
    "context": {
        "scope": "모든 계산은 같은 LOT 안에서 현재 분 이전까지의 값만 씀 (미래 값 없음)",
        "unit": "1분 (cal01_base_v4 이상의 1분 묶음)",
        "min_periods": 30,
        "ddof": 0,
        "z_threshold": 3,
        "window_5": 5,
        "window_10": 10,
        "common_sensors": COMMON_SENSORS,
        "common_ops": COMMON_OPS,
        "furnace_zones": FURNACE_ZONES,
        "drop_rows": "입력에 빈 값이 있는 행(LOT 앞부분)은 03과 같이 뺌",
        "source_notebook": "notebooks/sayletskin/03_model_f1.ipynb 6번 셀",
    },
}

with open(os.path.join(ZONE_DIR, "features.json"), "w", encoding="utf-8") as f:
    json.dump(features_doc, f, ensure_ascii=False, indent=1)

# 데이터 파일 해시 (어느 파일로 학습했는지)
sha = hashlib.sha256()
with open(BASE_CSV, "rb") as f:
    sha.update(f.read())

config = {
    "model_ids": MODEL_IDS,
    "label_version": LABEL_VERSION,
    "input_tag": input_tag,
    "input_tail": name_tail,
    "zone": ZONE,
    "zone_sensors": sensors,
    "data_file": DATA_FILE,
    "data_sha256": sha.hexdigest(),
    "switches": switch_values,
    "params": {
        "RandomForest": {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced", "random_state": 42},
        "IsolationForest": {"n_estimators": 100, "contamination": round(float(min(rate_all, 0.5)), 6), "random_state": 42},
    },
    "label_rule": "DOMAIN_LABEL이 있으면 그 값, 없으면 STAT_LABEL. EXCLUDED 행과 라벨 빈 행은 뺌",
    "rows": int(len(df)),
    "label_1_rate": round(float(rate_all), 6),
    "feature_count": len(features),
    "train_lots_all14": [int(x) for x in lots],
    "train_lots_fold13": [int(x) for x in lots[:13]],
    "fold13_test_lot": int(last_lot),
    "fold13_reproduced": bool(reproduced),
    "fold13_same_minutes": same_minutes,
    "fold13_total_minutes": total_minutes,
    "evidence": EVIDENCE.get(ZONE, "통계 후보만: 이 구역에는 팀 판단이 없음"),
    "sklearn": sklearn.__version__,
    "python": platform.python_version(),
    "created": TODAY,
}
with open(os.path.join(ZONE_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=1)

# 03 시간순 걷기 결과
summary = pd.read_csv(os.path.join(RESULT_DIR, result_name + "_summary.csv"), encoding=ENC)
metrics = {"source": result_name + "_summary.csv", "method": "시간순 걷기, 첫 4 LOT 학습, 시험 10번 합산"}
for _, row in summary.iterrows():
    metrics[row["모델"]] = {
        "F1": float(row["F1"]), "Precision": float(row["Precision"]), "Recall": float(row["Recall"]),
        "TP": int(row["TP"]), "FP": int(row["FP"]), "FN": int(row["FN"]), "LABEL_1": int(row["LABEL 1"]), "rows": int(row["시험 행"]),
    }
with open(os.path.join(ZONE_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=1)

# 재현 확인용 기대값 (센서 값은 넣지 않고 예측만)
expected = {"model_id": MODEL_IDS["rf_fold13"], "lot": int(last_lot), "minutes": []}
for _, row in compare.iterrows():
    expected["minutes"].append({"t": row["MEAS_DT"].strftime("%Y-%m-%d %H:%M:%S"), "pred": int(row["PRED_05"])})
with open(os.path.join(ZONE_DIR, "fold13_expected.json"), "w", encoding="utf-8") as f:
    json.dump(expected, f, ensure_ascii=False, indent=1)

# 모델 파일
joblib.dump(rf_all14, os.path.join(ZONE_DIR, "rf-all14-" + TODAY + ".joblib"))
joblib.dump(rf_fold13, os.path.join(ZONE_DIR, "rf-fold13-" + TODAY + ".joblib"))
joblib.dump(iso_all14, os.path.join(ZONE_DIR, "iso-all14-" + TODAY + ".joblib"))

print("저장:", sorted(os.listdir(ZONE_DIR)))

## 11. registry.json과 Z-score 설정

- `models/registry.json`에 이 구역의 모델 3개를 등록함. 같은 ID가 있으면 바꾸고, 없으면 더함
- Z-score는 파일이 없는 규칙이라 `models/<라벨 버전>/zscore.json`에 설정을 적고 같은 registry에 등록함
- `evidence`는 관제가 구역마다 "팀 라벨이 있는 모델인지"를 보여 주기 위한 칸

In [ ]:
REGISTRY = os.path.join(MODEL_DIR, "registry.json")
if os.path.isfile(REGISTRY):
    with open(REGISTRY, encoding="utf-8") as f:
        registry = json.load(f)
else:
    registry = {"models": []}

relative_zone_dir = LABEL_VERSION + "/" + input_tag + "/" + ZONE
rf_metrics = metrics.get("RandomForest", {})
iso_metrics = metrics.get("IsolationForest", {})

new_entries = [
    {"id": MODEL_IDS["rf_all14"], "kind": "RandomForest", "zone": ZONE, "label_version": LABEL_VERSION, "input_tag": input_tag,
     "path": relative_zone_dir + "/rf-all14-" + TODAY + ".joblib", "trained_on": "all14", "status": "active",
     "evidence": config["evidence"], "walk_forward_f1": rf_metrics.get("F1"), "walk_forward_fp": rf_metrics.get("FP"),
     "sklearn": sklearn.__version__, "created": TODAY, "features": relative_zone_dir + "/features.json", "config": relative_zone_dir + "/config.json"},
    {"id": MODEL_IDS["rf_fold13"], "kind": "RandomForest", "zone": ZONE, "label_version": LABEL_VERSION, "input_tag": input_tag,
     "path": relative_zone_dir + "/rf-fold13-" + TODAY + ".joblib", "trained_on": "fold13", "status": "check_only",
     "evidence": config["evidence"], "reproduced": bool(reproduced), "expected": relative_zone_dir + "/fold13_expected.json",
     "sklearn": sklearn.__version__, "created": TODAY, "features": relative_zone_dir + "/features.json", "config": relative_zone_dir + "/config.json"},
    {"id": MODEL_IDS["iso_all14"], "kind": "IsolationForest", "zone": ZONE, "label_version": LABEL_VERSION, "input_tag": input_tag,
     "path": relative_zone_dir + "/iso-all14-" + TODAY + ".joblib", "trained_on": "all14", "status": "active",
     "evidence": config["evidence"], "walk_forward_f1": iso_metrics.get("F1"), "walk_forward_fp": iso_metrics.get("FP"),
     "sklearn": sklearn.__version__, "created": TODAY, "features": relative_zone_dir + "/features.json", "config": relative_zone_dir + "/config.json"},
]

# Z-score 규칙 (라벨 버전마다 한 번)
zscore_path = os.path.join(VERSION_DIR, "zscore.json")
if not os.path.isfile(zscore_path):
    zscore_doc = {
        "id": "cal01-" + LABEL_VERSION + "-zscore-allzones",
        "rule": "같은 LOT 안에서 현재 분 이전까지의 값으로 expanding 평균과 표준편차를 구하고 |현재값 - 평균| / 표준편차 > 3이면 1",
        "min_periods": 30, "ddof": 0, "threshold": 3,
        "zone_rule": "구역 센서 중 하나라도 |Z| > 3이면 그 분을 1로 봄",
        "early_rows": "앞 30분은 기준이 부족해 0",
        "source_notebook": "notebooks/sayletskin/03_model_f1.ipynb 7번 셀",
    }
    with open(zscore_path, "w", encoding="utf-8") as f:
        json.dump(zscore_doc, f, ensure_ascii=False, indent=1)
    new_entries.append({"id": "cal01-" + LABEL_VERSION + "-zscore-allzones", "kind": "Z-score", "zone": "all", "label_version": LABEL_VERSION,
                        "input_tag": "none", "path": LABEL_VERSION + "/zscore.json", "trained_on": "none", "status": "active",
                        "evidence": "규칙형: 정답을 쓰지 않음. 팀 라벨이 없는 구역의 기본 감지", "created": TODAY})

kept = []
new_ids = []
for entry in new_entries:
    new_ids.append(entry["id"])
for entry in registry["models"]:
    if entry["id"] not in new_ids:
        kept.append(entry)
registry["models"] = kept + new_entries
registry["updated"] = TODAY

with open(REGISTRY, "w", encoding="utf-8") as f:
    json.dump(registry, f, ensure_ascii=False, indent=1)

print("registry 항목:", len(registry["models"]))
for entry in registry["models"]:
    print(" ", entry["id"], "|", entry["status"])